다음은 LangChain Hub 에서 프롬프트를 받아서 실행하는 예제입니다.

아래 주소에서 LangChain Hub 프롬프트를 확인할 수 있습니다. LangChain Hub는 사람들이 만들어 놓은 프롬프트 템플릿을 공유하는 공간입니다.

먼저 `LangcSmith`에 로그인을 합니다.

받아오는 방법은 프롬프트 repo 의 아이디 값을 가져 올 수 있고, commit id 를 붙여서 특정 버전에 대한 프롬프트를 받아올 수도 있습니다.


## Hub로부터 Prompt 받아오기


In [5]:
!pip install langsmith

최신 `LangSmith` 보안 정책 때문에 발생한 오류

In [ ]:
from langsmith import Client

#langsmith 서비스와 통신하기 위한 Client 객체 생성
client=Client()

prompt = client.pull_prompt(
    "rlm/rag-prompt",
    dangerously_pull_public_prompt=True
)

In [8]:
# 프롬프트 내용 출력
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


### rlm/rag-prompt 내용

You are an assistant for question-answering tasks.

# 검색된 참고 문서(examples x)를 활용하도록 지시
Use the following pieces of retrieved context to answer the question.

# 모르면 모른다고 답변
If you don't know the answer, just say that you don't know.

# 답변 길이를 최대 세 문장으로 간격하게 답변
Use three sentences maximum and keep the answer concise.

Question: {question}

Context: {context}

Answer:

### RAG 구현 순서

1. DOCX(참고문서)를 Chunk로 분할
2. OpenAI Embedding으로 벡터화
3. Chroma에 저장
4. Retriever로 관련 문서를 검색
5. LLM이 검색 결과를 참고하여 답변

In [ ]:
# RAG 구현 예시 코드
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# chain 생성
qa_chain=({
    "context":database.as_retriever(),
    "question":RunnablePassthrough(),
    }
    |prompt
    |llm
    |StrOutputParser()
)

# 실행
query = "연봉 5천만원 직장인의 소득세 계산"

result = qa_chain.invoke(query)

print(result)

핵심 개념 정리
| 구성요소             | 역할             |
| -------------------|-----------------|
|Loader | 문서읽기|
|Text Splitter  | 문서 분할(chunk 단위)|
|Embedding | 벡터 변환|
|Chroma | 벡터 저장|
|Retriever | 관련 문서 검색|
|Prompt | 질문 + 문맥 구성|
|LLM | prompt를 기반으로 최종 답변 생성|
|Output Parser | 결과 문자열 변환|

In [9]:
# 특정 버전의 프롬프트를 가져오려면 버전 해시를 지정하세요
prompt = client.pull("rlm/rag-prompt:50442af1")
prompt

AttributeError: 'Client' object has no attribute 'pull'

## Prompt Hub 에 자신의 프롬프트 등록


In [ ]:
from langchain.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(
    "주어진 내용을 바탕으로 다음 문장을 요약하세요. 답변은 반드시 한글로 작성하세요\n\nCONTEXT: {context}\n\nSUMMARY:"
)
prompt

In [ ]:
from langchain import hub

# 프롬프트를 허브에 업로드합니다.
hub.push("teddynote/simple-summary-korean", prompt)

다음은 Hub 에 성공적으로 업로드 된 후 출력입니다.

`아이디/프롬프트명/해시`


> 출력: 'https://smith.langchain.com/hub/teddynote/simple-summary-korean/0e296563'


In [ ]:
from langchain import hub

# 프롬프트를 허브로부터 가져옵니다.
pulled_prompt = hub.pull("teddynote/simple-summary-korean")

In [ ]:
# 프롬프트 내용 출력
print(pulled_prompt)